# Reprocessing Data

Improving the dataset by exploring alternative feature engineering and preprocessing methods to boost model performance.

## Import libraries

Loading essential libraries for data manipulation, visualization, modeling, and evaluation.

In [497]:
import numpy as np
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [498]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

---

## Initial Exploration

Performed a quick inspection of the dataset to understand structure, missing values, and data types.


In [499]:
raw_data = pd.read_csv('train.csv')

In [500]:
raw_data.sample(5)

,customer_id,Name,age,gender,security_no,region_category,membership_category,joining_date,joined_through_referral,referral_id,...,avg_time_spent,avg_transaction_value,avg_frequency_login_days,points_in_wallet,used_special_discount,offer_application_preference,past_complaint,complaint_status,feedback,churn_risk_score
30001,fffe43004900440035003500360038003900,Perry Joshi,43,M,0MZL8CK,NaN,Silver Membership,2017-08-21,No,xxxxxxxx,...,1217.026025,11586.82,10.0,NaN,Yes,No,Yes,Solved,Poor Website,4
4644,fffe43004900440032003100360033003800,Genoveva Chadburn,48,M,NMKKW1E,Town,Platinum Membership,2017-06-03,Yes,CID40921,...,563.017105,27662.60,Error,417.384594,Yes,Yes,No,Not Applicable,Quality Customer Care,1
21896,fffe43004900440035003600340031003600,Elsa Underhill,12,M,R8EAUXY,City,Gold Membership,2015-07-07,Yes,CID29144,...,231.090000,9692.61,10.0,NaN,No,Yes,No,Not Applicable,User Friendly Website,1
5577,fffe43004900440033003500370032003700,Dulcie Wess,29,M,FMLWGVO,NaN,No Membership,2015-09-18,?,xxxxxxxx,...,147.560000,15909.88,22.0,609.530000,No,Yes,Yes,Solved in Follow-up,Poor Website,5
21165,fffe43004900440033003300330033003300,Gay Resto,13,F,01BGHLT,City,Basic Membership,2015-05-20,Yes,CID14068,...,123.910000,2164.45,15.0,507.450000,Yes,No,Yes,No Information Available,Too many ads,-1


In [501]:
raw_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36992 entries, 0 to 36991
Data columns (total 25 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   customer_id                   36992 non-null  object 
 1   Name                          36992 non-null  object 
 2   age                           36992 non-null  int64  
 3   gender                        36992 non-null  object 
 4   security_no                   36992 non-null  object 
 5   region_category               31564 non-null  object 
 6   membership_category           36992 non-null  object 
 7   joining_date                  36992 non-null  object 
 8   joined_through_referral       36992 non-null  object 
 9   referral_id                   36992 non-null  object 
 10  preferred_offer_types         36704 non-null  object 
 11  medium_of_operation           36992 non-null  object 
 12  internet_option               36992 non-null  object 
 13  l

---

## Initial Processing

Handled missing values and basic data cleaning to prepare for deeper preprocessing.

In [502]:
drop_cols = ['customer_id', 'Name', 'security_no', 'referral_id', 'last_visit_time']

data = raw_data.drop(drop_cols, axis=1)

In [503]:
placeholders = ['?', 'NA', 'null', 'n/a', '-',
                'unknown', 'Unknown', 'UNKNOWN', 'Error'
                , 'No reason specified']
data = data.replace(placeholders, np.nan)

In [504]:
data['avg_frequency_login_days'] = data['avg_frequency_login_days'].astype(float)

In [505]:
data['joining_date'] = pd.to_datetime(data['joining_date'], errors='coerce')

today = pd.to_datetime(datetime.today())
data['years_since_joining'] = (today - data['joining_date']).dt.days // 365
data = data.drop('joining_date', axis=1)

In [506]:
data['churn_risk_score'].value_counts()

,count
churn_risk_score,
3,10424
4,10185
5,9827
2,2741
1,2652
-1,1163


In [507]:
data['churn_risk_score'] = data['churn_risk_score'].replace(-1, np.nan)
data = data.dropna(subset=['churn_risk_score'])

In [508]:
membership_map = {
    'No Membership': 0,
    'Basic Membership': 1,
    'Silver Membership': 2,
    'Gold Membership': 3,
    'Premium Membership': 4,
    'Platinum Membership': 5
}

data['membership_category_encoded'] = data['membership_category'].map(membership_map)
data = data.drop('membership_category', axis=1)

In [509]:
# Handling feedback type
def feedback_type(x):
  good_feedback = ['Quality Customer Care', 'Products always in Stock',
                   'User Friendly Website', 'Reasonable Price']
  bad_feedback = ['Poor Product Quality', 'Too many ads', 'Poor Customer Service']
  if x in good_feedback:
    return 'good'
  elif x in bad_feedback:
    return 'bad'

data['feedback'] = data['feedback'].apply(feedback_type)

In [510]:
# Handling Complaint Status
status_map = {
    'Solved': 'Solved',
    'Solved in Follow-up': 'Solved',
    'Unsolved': 'Unsolved',
    'Not Applicable': 'Unsolved',
    'No Information Available': 'Unsolved'
}

data['complaint_status'] = data['complaint_status'].map(status_map)

In [511]:
data.isnull().sum()

,0
age,0
gender,56
region_category,5263
joined_through_referral,5292
preferred_offer_types,276
medium_of_operation,5230
internet_option,0
days_since_last_login,0
avg_time_spent,0
avg_transaction_value,0


In [512]:
data = data.dropna(subset=['churn_risk_score'])
data['churn_risk_score'] = data['churn_risk_score'].astype(int)

In [513]:
X = data.drop('churn_risk_score', axis=1)
y = data['churn_risk_score']

In [514]:
num_data = X.select_dtypes(exclude='object')
cat_data = X.select_dtypes(include='object')

cat_cols = cat_data.columns
num_cols = num_data.columns

---

## Handling categorical variables

Processed categorical features strategically to maintain information while optimizing for model compatibility.


In [515]:
binary_cols = [col for col in cat_cols if cat_data[col].nunique() == 2]
nominal_cols = [col for col in cat_cols if cat_data[col].nunique() > 2]

### Handling Binary Columns

Encoded binary features using label encoding to retain clarity with minimal transformation.


In [516]:
binary_data = data[binary_cols]

In [517]:
binary_cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

### Handling Nominal Columns

Applied one-hot encoding to nominal features to enable machine learning models to interpret them properly.


In [518]:
nominal_data = data[nominal_cols]

In [519]:
nominal_cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder())
])

---

## Handling Numerical Data

In [520]:
num_data_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', MinMaxScaler())
])

---

## Transforming Data

Scaled numerical features and applied transformations where necessary to standardize input for modeling.

In [521]:
preprocessor = ColumnTransformer([
    ('num', num_data_pipe, num_cols),
    ('binary', binary_cat_pipe, binary_cols),
    ('nominal', nominal_cat_pipe, nominal_cols)
    ])

---

## Model Training

Trained the model using XGBoost with the updated pipeline and refined features.

In [522]:
from sklearn.metrics import accuracy_score, classification_report, recall_score, precision_score, f1_score
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

In [523]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [524]:
y_train = y_train-1
y_test = y_test-1

In [525]:
XG = XGBClassifier(random_state=42, num_class=5)

model_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XG)
])

---

## Fine Tuning Hyperparameters

Applied randomized search to optimize key hyperparameters for better performance without high computational cost.


In [526]:
param_dist = {
    'model__learning_rate': np.linspace(0.01, 0.2, 5),
    'model__max_depth': [3, 5, 7, 9],
    'model__min_child_weight': [1, 3, 5],
    'model__subsample': [0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.6, 0.7, 0.8, 1.0],
    'model__n_estimators': [50, 100, 200],
    'model__gamma': [0, 0.1, 0.2]
}

In [527]:
random_search = RandomizedSearchCV(
    estimator= model_pipe,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=2,
    random_state=42
)

random_search.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer()),
                                                                                               ('scaler',
                                                                                                MinMaxScaler())]),
                                                                               Index(['age', 'days_since_last_login', 'avg_time_spent',
       'avg_transaction_value', 'avg_frequency_login_days', 'points_in_wallet',
       'years_since_joining', 'membership_category_encode...
                                                            num_parallel_tree=None, ...))]),
                   n_jobs=-1,
                   param_distributions={'model__colsample_bytree': [0.6, 0.7,
                                                                    0.8, 1.0],
                                        'model__gamma': [0, 0.1, 0.2],
                                        'model__learning_rate': array([0.01  , 0.0575, 0.105 , 0.1525, 0.2   ]),
                                        'model__max_depth': [3, 5, 7, 9],
                                        'model__min_child_weight': [1, 3, 5],
                                        'model__n_estimators': [50, 100, 200],
                                        'model__subsample': [0.7, 0.8, 0.9,
                                                             1.0]},
                   random_state=42, scoring='f1_macro', verbose=2)

---

## Results

Achieved improved evaluation metrics, demonstrating the effectiveness of the revised data handling pipeline.


In [528]:
train_y_pred = random_search.predict(X_train)
test_y_pred = random_search.predict(X_test)

In [529]:
print(accuracy_score(y_train, train_y_pred))
print(precision_score(y_train, train_y_pred, average='macro'))
print(recall_score(y_train, train_y_pred, average='macro'))
print(f1_score(y_train, train_y_pred, average='macro'))
print(classification_report(y_train, train_y_pred))

0.7856470013606391
0.8128751268560078
0.7855105042474159
0.7736243111994047
              precision    recall  f1-score   support

           0       0.69      0.97      0.81      2107
           1       0.96      0.58      0.73      2200
           2       0.95      0.86      0.90      8296
           3       0.78      0.51      0.62      8181
           4       0.69      1.00      0.81      7879

    accuracy                           0.79     28663
   macro avg       0.81      0.79      0.77     28663
weighted avg       0.81      0.79      0.78     28663



In [530]:
print(accuracy_score(y_test, test_y_pred))
print(precision_score(y_test, test_y_pred, average='macro'))
print(recall_score(y_test, test_y_pred, average='macro'))
print(f1_score(y_test, test_y_pred, average='macro'))
print(classification_report(y_test, test_y_pred))

0.7972369522746302
0.8217756984457878
0.7964362983743588
0.7871256010242549
              precision    recall  f1-score   support

           0       0.72      0.97      0.83       545
           1       0.95      0.62      0.75       541
           2       0.96      0.87      0.91      2128
           3       0.79      0.52      0.63      2004
           4       0.69      1.00      0.82      1948

    accuracy                           0.80      7166
   macro avg       0.82      0.80      0.79      7166
weighted avg       0.82      0.80      0.79      7166



---

## Saving the model

Exported the trained model pipeline for future use and reproducibility.

In [531]:
import joblib as jb

jb.dump(random_search, 'xg_model.joblib')

['xg_model.joblib']